# Bank Customer Churn Model Training

This notebook trains multiple machine learning models for the Bank Customer Churn dataset and evaluates their performance. It includes Random Forest, XGBoost, LightGBM, and a hybrid stacking ensemble to compare predictive performance.

## 1. Import Libraries

We import the libraries required for model training, evaluation, and serialization.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Optional imports for gradient boosting libraries
try:
    import xgboost as xgb
except ImportError:
    xgb = None

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Preprocessed Data

The notebook loads the processed training and testing files generated during preprocessing.

In [2]:
# Define paths to processed files
processed_dir = Path("../datasets/processed")

X_train = pd.read_csv(processed_dir / "X_train_scaled.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv").iloc[:, 0]
X_test = pd.read_csv(processed_dir / "X_test_scaled.csv")
y_test = pd.read_csv(processed_dir / "y_test.csv").iloc[:, 0]

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (8000, 11)
Testing data shape: (2000, 11)


## 3. Train Random Forest

A Random Forest classifier is trained as a strong tree-based baseline model.

In [3]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_metrics = {
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1": f1_score(y_test, rf_pred),
    "ROC-AUC": roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])
}

print("Random Forest Metrics:")
print(rf_metrics)

Random Forest Metrics:
{'Accuracy': 0.864, 'Precision': 0.7824267782426778, 'Recall': 0.4594594594594595, 'F1': 0.5789473684210527, 'ROC-AUC': 0.8520762673305046}


## 4. Train XGBoost

XGBoost is trained as a gradient boosting model for improved predictive power.

In [4]:
if xgb is None:
    print("XGBoost is not installed. Skipping training.")
    xgb_model = None
else:
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
    )
    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_test)

    xgb_metrics = {
        "Accuracy": accuracy_score(y_test, xgb_pred),
        "Precision": precision_score(y_test, xgb_pred),
        "Recall": recall_score(y_test, xgb_pred),
        "F1": f1_score(y_test, xgb_pred),
        "ROC-AUC": roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])
    }

    print("XGBoost Metrics:")
    print(xgb_metrics)

XGBoost Metrics:
{'Accuracy': 0.867, 'Precision': 0.7701149425287356, 'Recall': 0.49385749385749383, 'F1': 0.6017964071856288, 'ROC-AUC': 0.8645224577427967}


C:\Users\MITHRA\RetainIQ\backend\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:14:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## 5. Train LightGBM

LightGBM is trained as another gradient boosting model to compare performance.

In [5]:
if lgb is None:
    print("LightGBM is not installed. Skipping training.")
    lgb_model = None
else:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    )
    lgb_model.fit(X_train, y_train)
    lgb_pred = lgb_model.predict(X_test)

    lgb_metrics = {
        "Accuracy": accuracy_score(y_test, lgb_pred),
        "Precision": precision_score(y_test, lgb_pred),
        "Recall": recall_score(y_test, lgb_pred),
        "F1": f1_score(y_test, lgb_pred),
        "ROC-AUC": roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1])
    }

    print("LightGBM Metrics:")
    print(lgb_metrics)

[LightGBM] [Info] Number of positive: 1630, number of negative: 6370
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000116 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 864
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203750 -> initscore=-1.363019
[LightGBM] [Info] Start training from score -1.363019
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

## 6. Build Hybrid Stacking Ensemble

A stacking ensemble is created using the trained base models and a logistic regression meta-learner.

In [6]:
models = [
    ("rf", rf_model)
]

if xgb_model is not None:
    models.append(("xgb", xgb_model))

if lgb_model is not None:
    models.append(("lgb", lgb_model))

if len(models) < 2:
    print("Not enough models available to build a stacking ensemble.")
else:
    stack_model = StackingClassifier(
        estimators=models,
        final_estimator=LogisticRegression(random_state=42),
        cv=3,
        n_jobs=-1
    )

    stack_model.fit(X_train, y_train)
    print(type(stack_model))
    print(stack_model)
    stack_pred = stack_model.predict(X_test)

    stack_metrics = {
        "Accuracy": accuracy_score(y_test, stack_pred),
        "Precision": precision_score(y_test, stack_pred),
        "Recall": recall_score(y_test, stack_pred),
        "F1": f1_score(y_test, stack_pred),
        "ROC-AUC": roc_auc_score(y_test, stack_model.predict_proba(X_test)[:, 1])
    }

    print("Stacking Ensemble Metrics:")
    print(stack_metrics)

<class 'sklearn.ensemble._stacking.StackingClassifier'>
StackingClassifier(cv=3,
                   estimators=[('rf',
                                RandomForestClassifier(n_estimators=200,
                                                       n_jobs=-1,
                                                       random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=True,
                                              eval_metric='logloss',
         

## 7. Compare Model Performance

We compare the metrics of the trained models in a single summary table.

In [7]:
results = {
    "Random Forest": rf_metrics
}

if xgb_model is not None:
    results["XGBoost"] = xgb_metrics

if lgb_model is not None:
    results["LightGBM"] = lgb_metrics

if "stack_metrics" in locals():
    results["Stacking Ensemble"] = stack_metrics

comparison_df = pd.DataFrame(results).T
print(comparison_df)

                   Accuracy  Precision    Recall        F1   ROC-AUC
Random Forest        0.8640   0.782427  0.459459  0.578947  0.852076
XGBoost              0.8670   0.770115  0.493857  0.601796  0.864522
LightGBM             0.8640   0.762646  0.481572  0.590361  0.865232
Stacking Ensemble    0.8695   0.789683  0.488943  0.603945  0.863148


## 8. Save the Trained Model

The best-performing model is saved for later use in prediction or deployment workflows.

In [8]:
# Create deployment artifact directory
project_root = Path.cwd()
backend_root = project_root / "backend" if (project_root / "backend").exists() else project_root
if (project_root.parent / "backend").exists() and not (project_root / "backend").exists():
    backend_root = project_root.parent / "backend"

saved_models_dir = backend_root / "saved_models"
saved_models_dir.mkdir(parents=True, exist_ok=True)

# Save the trained stacking ensemble, falling back to the random forest model if needed
model_to_save = stack_model if "stack_model" in locals() else rf_model
model_path = saved_models_dir / "stacking_model.pkl"

print("Saving:", type(model_to_save))
print("Path:", model_path)
joblib.dump(model_to_save, model_path)

print("Deployment artifact saved successfully.")
print("- Saved:", model_path.resolve())

Saving: <class 'sklearn.ensemble._stacking.StackingClassifier'>
Path: C:\Users\MITHRA\RetainIQ\backend\notebooks\saved_models\stacking_model.pkl
Deployment artifact saved successfully.
- Saved: C:\Users\MITHRA\RetainIQ\backend\notebooks\saved_models\stacking_model.pkl
